In [ ]:
# Download the missing Jan-Jun 2026 Welt articles (past the original END_YEAR=2025/END_MONTH=12
# cutoff in your code), using the same monthly-sitemap + category-filter + JSON-LD extraction
# method as your original scrape, then filter to Russia-in-title using your actual keyword list.

import re
import json
import gzip
import requests
import pandas as pd

from io import BytesIO
from lxml import etree
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# SETTINGS
# =====================================================

MONTHS_TO_FETCH = [
    (2026, 1), (2026, 2), (2026, 3), (2026, 4), (2026, 5), (2026, 6),
]

CUTOFF_DATE = pd.Timestamp("2026-06-11 23:59:59", tz="UTC")

MAX_WORKERS = 10

OUTPUT_FILE = "welt_2026_extension.csv"
FAILED_FILE = "welt_2026_extension_failed.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
}

ALLOWED_CATEGORIES = {
    "politik", "wirtschaft", "debatte", "vermischtes", "wissenschaft", "geschichte",
}

russia_keywords_de = re.compile(
    r"\b("
    r"russland\w*|russisch\w*|kreml\w*|putin\w*|moskau\w*|"
    r"lawrow\w*|lavrov\w*|peskow\w*|peskov\w*|schoigu\w*|shoigu\w*|"
    r"gerassimow\w*|gerasimow\w*|medwedew\w*|sacharowa\w*|mischustin\w*|"
    r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
    r"donezk\w*|luhansk\w*|krim\w*|crimea\w*"
    r")\b",
    re.IGNORECASE,
)

# =====================================================
# SITEMAP LOADING
# =====================================================

def load_urls(year, month):
    sitemap_url = f"https://www.welt.de/sitemaps/sitemap/{year}/{month:02d}/sitemap.xml.gz"
    try:
        r = requests.get(sitemap_url, headers=HEADERS, timeout=60)
        if r.status_code != 200:
            print(f"  {year}-{month:02d}: status {r.status_code}")
            return []
        xml_data = gzip.decompress(r.content)
        tree = etree.parse(BytesIO(xml_data))
        ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}
        return tree.xpath("//sm:url/sm:loc/text()", namespaces=ns)
    except Exception as e:
        print(f"  {year}-{month:02d}: ERROR {e}")
        return []

def filter_by_category(urls):
    result = []
    for url in urls:
        try:
            path = url.replace("https://www.welt.de/", "")
            category = path.split("/")[0]
            if category in ALLOWED_CATEGORIES:
                result.append((url, category))
        except Exception:
            pass
    return result

# =====================================================
# ARTICLE EXTRACTION (title + date + text from JSON-LD)
# =====================================================

def extract_article(url_category):
    url, category = url_category
    try:
        r = requests.get(url, headers=HEADERS, timeout=60)
        if r.status_code != 200:
            return (False, {"url": url, "category": category, "error": f"HTTP_{r.status_code}"})

        soup = BeautifulSoup(r.text, "lxml")
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.get_text(strip=True))
                items = data if isinstance(data, list) else [data]
                for item in items:
                    if isinstance(item, dict) and item.get("@type") == "NewsArticle":
                        title = item.get("headline")
                        date = item.get("datePublished")
                        text = item.get("articleBody")
                        if not text or not title:
                            continue
                        return (True, {
                            "article_url": url,
                            "category": category,
                            "title": title,
                            "date": date,
                            "text": text,
                        })
            except Exception:
                continue
        return (False, {"url": url, "category": category, "error": "NO_NEWSARTICLE_JSONLD"})
    except Exception as e:
        return (False, {"url": url, "category": category, "error": str(e)})

# =====================================================
# COLLECT URLS FOR TARGET MONTHS
# =====================================================

all_pairs = []
for year, month in MONTHS_TO_FETCH:
    print(f"loading {year}-{month:02d}...")
    urls = load_urls(year, month)
    filtered = filter_by_category(urls)
    print(f"  {len(urls)} total, {len(filtered)} in allowed categories")
    all_pairs.extend(filtered)

all_pairs = list({u: c for u, c in all_pairs}.items())
print(f"\ntotal unique URLs to fetch: {len(all_pairs)}")

# =====================================================
# FETCH ARTICLES
# =====================================================

success_rows = []
failed_rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(extract_article, pair) for pair in all_pairs]
    for future in tqdm(as_completed(futures), total=len(futures)):
        ok, result = future.result()
        (success_rows if ok else failed_rows).append(result)

df = pd.DataFrame(success_rows)
print(f"\nfetched: {len(df)}, failed: {len(failed_rows)}")

# =====================================================
# FILTER: date within window + Russia in title
# =====================================================

df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df = df[df["date_parsed"] <= CUTOFF_DATE]

df["title_has_russia"] = df["title"].fillna("").str.contains(russia_keywords_de, na=False)
df_russia = df[df["title_has_russia"]].drop(columns=["title_has_russia", "date_parsed"])

print(f"\narticles with Russia in title: {len(df_russia)}")

df_russia.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
if failed_rows:
    pd.DataFrame(failed_rows).to_csv(FAILED_FILE, index=False, encoding="utf-8-sig")

print(f"\nsaved: {OUTPUT_FILE}")

После прогона у вас будет welt_2026_extension.csv с теми же по смыслу колонками (article_url, category, title, date, text), только без image_url/image_filename — картинки нужно будет докачать отдельным шагом (та же логика, что в cell 21-25 вашего ноутбука: собрать image_urls со страницы, дедуплицировать, скачать).

In [ ]:
# =====================================================
# 1) Download images for welt_2026_extension.csv (409 articles)
# 2) Save them into welt_final_images with the SAME naming scheme
#    already used for the rest of the welt dataset (asset-id.jpg,
#    the 32-char hex id embedded in the welt.de CDN image URL)
# 3) Merge the 409 new rows into welt_final_data.csv
# =====================================================

import re
import json
import requests
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# SETTINGS -- adjust paths if needed
# =====================================================

EXTENSION_FILE = "welt_2026_extension.csv"          # output of your previous script
FINAL_DATA_FILE = "welt_final_data.csv"              # existing full dataset
FINAL_DATA_BACKUP = "welt_final_data_backup.csv"     # safety backup, written automatically
IMAGES_DIR = Path(".")   

In [ ]:
            # existing images folder (same one you already have)

MAX_WORKERS = 10

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
}

# matches the 32-char hex asset id embedded in welt.de CDN image urls,
# e.g. https://images.welt.de/<docid>/<ASSETID>/ci16x9-w1200/<slug>
ASSET_ID_RE = re.compile(r"/([a-f0-9]{32})/ci\d")

IMAGES_DIR.mkdir(exist_ok=True)

# =====================================================
# STEP 1: for each article, re-fetch the page and pull the image field
#         from the same NewsArticle JSON-LD block used for text/title
# =====================================================

def extract_image(article_url):
    try:
        r = requests.get(article_url, headers=HEADERS, timeout=60)
        if r.status_code != 200:
            return {"article_url": article_url, "image_url": None, "image_filename": None,
                     "error": f"HTTP_{r.status_code}"}

        soup = BeautifulSoup(r.text, "lxml")
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.get_text(strip=True))
            except Exception:
                continue
            items = data if isinstance(data, list) else [data]
            for item in items:
                if isinstance(item, dict) and item.get("@type") == "NewsArticle":
                    img = item.get("image")
                    if not img:
                        continue
                    # JSON-LD "image" can be a single string, a dict, or a list of
                    # strings/dicts (one per crop variant e.g. ci16x9 / ci102l / ci3x2l)
                    if isinstance(img, dict):
                        img = [img.get("url")]
                    elif isinstance(img, str):
                        img = [img]
                    elif isinstance(img, list):
                        img = [i.get("url") if isinstance(i, dict) else i for i in img]
                    img = [u for u in img if u]
                    if not img:
                        continue

                    image_url_field = "|".join(img)

                    filename = None
                    for u in img:
                        m = ASSET_ID_RE.search(u)
                        if m:
                            filename = m.group(1) + ".jpg"
                            break

                    return {"article_url": article_url, "image_url": image_url_field,
                             "image_filename": filename, "error": None}

        return {"article_url": article_url, "image_url": None, "image_filename": None,
                 "error": "NO_IMAGE_FOUND"}
    except Exception as e:
        return {"article_url": article_url, "image_url": None, "image_filename": None,
                 "error": str(e)}


def download_image(image_url_field, filename):
    """Downloads the first (ci16x9-w1200) crop variant, matching what's already
    on disk for the rest of the dataset."""
    if pd.isna(image_url_field) or pd.isna(filename):
        return False
    dest = IMAGES_DIR / filename
    if dest.exists():
        return True  # already have it
    first_url = image_url_field.split("|")[0]
    try:
        r = requests.get(first_url, headers=HEADERS, timeout=60)
        if r.status_code == 200 and r.content:
            dest.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False


# =====================================================
# RUN
# =====================================================

ext = pd.read_csv(EXTENSION_FILE)
print(f"loaded {len(ext)} rows from {EXTENSION_FILE}")

image_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(extract_image, url) for url in ext["article_url"]]
    for future in tqdm(as_completed(futures), total=len(futures)):
        image_results.append(future.result())

img_df = pd.DataFrame(image_results)
print(f"\nimages found: {img_df['image_filename'].notna().sum()} / {len(img_df)}")
print("\nreasons for missing images:")
print(img_df.loc[img_df["image_filename"].isna(), "error"].fillna("(no error, but no image field)").value_counts())

# download the actual image files
downloaded = 0
for _, row in tqdm(img_df.iterrows(), total=len(img_df), desc="downloading"):
    if download_image(row["image_url"], row["image_filename"]):
        downloaded += 1
print(f"\nimages downloaded/already present: {downloaded}")

In [ ]:
# =====================================================
# MERGE image_url / image_filename into the extension data
# =====================================================

ext_merged = ext.merge(
    img_df[["article_url", "image_url", "image_filename"]],
    on="article_url", how="left"
)

# keep only the columns used in welt_final_data.csv, same order
ext_merged = ext_merged[["category", "article_url", "title", "date", "text",
                           "image_url", "image_filename"]]

# =====================================================
# BACKUP + APPEND into welt_final_data.csv
# =====================================================

final = pd.read_csv(FINAL_DATA_FILE)
final.to_csv(FINAL_DATA_BACKUP, index=False, encoding="utf-8-sig")
print(f"\nbackup written: {FINAL_DATA_BACKUP} ({len(final)} rows)")

before = len(final)
combined = pd.concat([final, ext_merged], ignore_index=True)
combined = combined.drop_duplicates(subset="article_url", keep="first")
print(f"final: {before} -> {len(combined)} rows ({len(combined) - before} new)")

combined.to_csv(FINAL_DATA_FILE, index=False, encoding="utf-8-sig")
print(f"\nsaved: {FINAL_DATA_FILE}")

# also save the extension file with images filled in, for your own records
ext_merged.to_csv("welt_2026_extension_with_images.csv", index=False, encoding="utf-8-sig")
print("saved: welt_2026_extension_with_images.csv")

In [ ]:
import json
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
}

url = "https://www.welt.de/politik/ausland/plus69bd2fa9e5e9d619745d14a3/putin-greift-nach-der-letzten-freiheit-der-russen.html"

r = requests.get(url, headers=HEADERS, timeout=60)
soup = BeautifulSoup(r.text, "lxml")

print("=== все @type в JSON-LD на странице ===")
for script in soup.find_all("script", type="application/ld+json"):
    try:
        data = json.loads(script.get_text(strip=True))
    except Exception:
        continue
    items = data if isinstance(data, list) else [data]
    for item in items:
        if isinstance(item, dict):
            print(" -", item.get("@type"), "| ключи:", list(item.keys()))

print("\n=== og:image ===")
og = soup.find("meta", property="og:image")
print(og["content"] if og else "NOT FOUND")

print("\n=== twitter:image ===")
tw = soup.find("meta", attrs={"name": "twitter:image"})
print(tw["content"] if tw else "NOT FOUND")

In [ ]:
import re
import json
import requests
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# SETTINGS
# =====================================================

EXT_WITH_IMAGES_FILE = "welt_2026_extension_with_images.csv"
FINAL_DATA_FILE = "welt_final_data.csv"
FINAL_DATA_BACKUP2 = "welt_final_data_backup2.csv"
IMAGES_DIR = Path("welt_final_images")

MAX_WORKERS = 10

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
}

# 32-char hex asset id, optionally followed by a suffix like "-master",
# right before the /ci16x9-w1200/ crop segment
ASSET_ID_RE = re.compile(r"/([a-f0-9]{32})(?:-\w+)?/ci\d")

IMAGES_DIR.mkdir(exist_ok=True)

# =====================================================
# IMPROVED extractor: NewsArticle "image" (url OR contentUrl,
# single dict/string/list) -> separate ImageObject block on the
# page -> og:image meta tag, in that order
# =====================================================

def get_url_from_image_field(img):
    """Normalizes the JSON-LD 'image' field into a list of URL strings,
    whatever shape it comes in (string / dict / list of either)."""
    if img is None:
        return []
    if isinstance(img, str):
        return [img]
    if isinstance(img, dict):
        u = img.get("url") or img.get("contentUrl")
        return [u] if u else []
    if isinstance(img, list):
        out = []
        for i in img:
            if isinstance(i, str):
                out.append(i)
            elif isinstance(i, dict):
                u = i.get("url") or i.get("contentUrl")
                if u:
                    out.append(u)
        return out
    return []


def extract_image(article_url):
    try:
        r = requests.get(article_url, headers=HEADERS, timeout=60)
        if r.status_code != 200:
            return {"article_url": article_url, "image_url": None, "image_filename": None,
                     "error": f"HTTP_{r.status_code}"}

        soup = BeautifulSoup(r.text, "lxml")

        ld_blocks = []
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.get_text(strip=True))
            except Exception:
                continue
            items = data if isinstance(data, list) else [data]
            ld_blocks.extend([i for i in items if isinstance(i, dict)])

        # 1) try the "image" field on the NewsArticle block itself
        urls = []
        for item in ld_blocks:
            if item.get("@type") == "NewsArticle":
                urls = get_url_from_image_field(item.get("image"))
                if urls:
                    break

        # 2) fall back to a standalone ImageObject block elsewhere on the page
        if not urls:
            for item in ld_blocks:
                if item.get("@type") == "ImageObject":
                    u = item.get("contentUrl") or item.get("url")
                    if u:
                        urls = [u]
                        break

        # 3) fall back to og:image meta tag
        if not urls:
            og = soup.find("meta", property="og:image")
            if og and og.get("content"):
                urls = [og["content"]]

        if not urls:
            return {"article_url": article_url, "image_url": None, "image_filename": None,
                     "error": "NO_IMAGE_FOUND"}

        image_url_field = "|".join(urls)

        filename = None
        for u in urls:
            m = ASSET_ID_RE.search(u)
            if m:
                filename = m.group(1) + ".jpg"
                break

        if filename is None:
            return {"article_url": article_url, "image_url": image_url_field, "image_filename": None,
                     "error": "URL_FOUND_BUT_NO_ASSET_ID"}

        return {"article_url": article_url, "image_url": image_url_field,
                 "image_filename": filename, "error": None}
    except Exception as e:
        return {"article_url": article_url, "image_url": None, "image_filename": None,
                 "error": str(e)}


def download_image(image_url_field, filename):
    if pd.isna(image_url_field) or pd.isna(filename):
        return False
    dest = IMAGES_DIR / filename
    if dest.exists():
        return True
    first_url = str(image_url_field).split("|")[0]
    try:
        r = requests.get(first_url, headers=HEADERS, timeout=60)
        if r.status_code == 200 and r.content:
            dest.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False


# =====================================================
# RUN: only re-try the rows that are still missing an image
# =====================================================

ext = pd.read_csv(EXT_WITH_IMAGES_FILE)
missing = ext[ext["image_filename"].isna()].copy()
print(f"retrying {len(missing)} articles still missing an image")

retry_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(extract_image, url) for url in missing["article_url"]]
    for future in tqdm(as_completed(futures), total=len(futures)):
        retry_results.append(future.result())

retry_df = pd.DataFrame(retry_results)
print(f"\nnewly found: {retry_df['image_filename'].notna().sum()} / {len(retry_df)}")
print("\nremaining reasons for missing images:")
print(retry_df.loc[retry_df["image_filename"].isna(), "error"].value_counts())

# download the newly found images
downloaded = 0
for _, row in tqdm(retry_df.iterrows(), total=len(retry_df), desc="downloading"):
    if download_image(row["image_url"], row["image_filename"]):
        downloaded += 1
print(f"\nimages downloaded/already present: {downloaded}")

# =====================================================
# UPDATE ext (welt_2026_extension_with_images.csv) with the new results
# =====================================================

retry_df_idx = retry_df.set_index("article_url")
ext = ext.set_index("article_url")
for col in ["image_url", "image_filename"]:
    ext.loc[retry_df_idx.index, col] = ext.loc[retry_df_idx.index, col].combine_first(retry_df_idx[col])
ext = ext.reset_index()

ext.to_csv(EXT_WITH_IMAGES_FILE, index=False, encoding="utf-8-sig")
print(f"\nupdated: {EXT_WITH_IMAGES_FILE} (now {ext['image_filename'].notna().sum()} / {len(ext)} with images)")

# =====================================================
# UPDATE welt_final_data.csv (these 409 rows are already in there,
# but with the old, mostly-empty image_url/image_filename) -- backfill them
# =====================================================

final = pd.read_csv(FINAL_DATA_FILE)
final.to_csv(FINAL_DATA_BACKUP2, index=False, encoding="utf-8-sig")
print(f"backup written: {FINAL_DATA_BACKUP2} ({len(final)} rows)")

final = final.set_index("article_url")
ext_idx = ext.set_index("article_url")
for col in ["image_url", "image_filename"]:
    final.loc[ext_idx.index, col] = final.loc[ext_idx.index, col].combine_first(ext_idx[col])
final = final.reset_index()
# restore original column order
final = final[["category", "article_url", "title", "date", "text", "image_url", "image_filename"]]

final.to_csv(FINAL_DATA_FILE, index=False, encoding="utf-8-sig")
print(f"saved: {FINAL_DATA_FILE} ({final['image_filename'].notna().sum()} / {len(final)} rows with images)")